Import neccessary libraries

In [ ]:
import yaml
from pathlib import Path
import logging

import pandas as pd
import optuna
import mlflow
import mlflow.sklearn
import mlflow.lightgbm
import mlflow.xgboost

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. THIẾT LẬP LOGGING (Dùng force=True để Jupyter không bị lỗi khi chạy lại cell nhiều lần)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True
)
logger = logging.getLogger(__name__)

# 2. ĐƯỜNG DẪN DỮ LIỆU
CLEANED_DATA_PATH = Path("../data/processed/cleaned_data.csv")
CONFIG_PATH = Path("../artifacts/models/best_model_config.yaml")
TARGET_COLUMN = "churn_status"
SPLIT_COLUMN = "data_split"
RANDOM_STATE = 42

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Netflix_All_Models_Tuning")
# autolog can create many extra logs in notebook runs
# Keep it off for cleaner manual logging
# mlflow.sklearn.autolog()

overall_results = {}

2026/04/11 15:37:58 INFO mlflow.tracking.fluent: Experiment with name 'Netflix_All_Models_Tuning' does not exist. Creating a new experiment.


In [8]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [9]:
# Basic validation
required_cols = {TARGET_COLUMN, SPLIT_COLUMN}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if not set(df[SPLIT_COLUMN].unique()).issuperset({"train", "test"}):
    raise ValueError("Column 'data_split' must contain both 'train' and 'test' values.")

In [10]:
# =========================================
# Split data
# =========================================
train_set = df[df[SPLIT_COLUMN] == "train"].copy()
test_set = df[df[SPLIT_COLUMN] == "test"].copy()

if train_set.empty:
    raise ValueError("Train set is empty.")
if test_set.empty:
    raise ValueError("Test set is empty.")

X_train = train_set.drop(columns=[TARGET_COLUMN, SPLIT_COLUMN])
y_train = train_set[TARGET_COLUMN]

X_test = test_set.drop(columns=[TARGET_COLUMN, SPLIT_COLUMN])
y_test = test_set[TARGET_COLUMN]

# Cross-validation object for tuning on train set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### BASELINE

In [50]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="Baseline_LogisticRegression"):
    baseline_model = LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight="balanced",
        max_iter=1000
    )
    baseline_model.fit(X_train, y_train)

    y_test_proba = baseline_model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_metrics = {
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "test_log_loss": float(log_loss(y_test, y_test_proba)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
    }

    mlflow.log_params({
        "model_type": "LogisticRegression",
        "class_weight": "balanced",
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
    })
    mlflow.log_metrics(test_metrics)

    overall_results["Baseline_LogisticRegression"] = {
        "model": baseline_model,
        "cv_pr_auc": None,
        "test_pr_auc": float(test_metrics["test_pr_auc"]),
        "params": {
            "class_weight": "balanced",
            "max_iter": 1000,
            "random_state": RANDOM_STATE,
        },
    }

🏃 View run Baseline_LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/53a23953afbc494fb563f8d5f6180942
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### RANDOM FOREST

In [11]:
def objective_rf(trial):
    """
    Tune RandomForest using cross-validated PR-AUC on the training set only.
    This avoids test leakage during hyperparameter search.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 5, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy"]),
        "random_state": RANDOM_STATE,
        "class_weight": "balanced",
        "n_jobs": -1,
    }

    model = RandomForestClassifier(**params)

    # Out-of-fold predicted probabilities on training data
    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Default classification threshold for reporting auxiliary metrics
    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train, y_pred_oof)),
        "cv_precision": float(precision_score(y_train, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train, y_pred_oof, zero_division=0)),
    }

    # Log each Optuna trial as a nested MLflow run
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="RandomForest_PR_AUC_Optimization_CV"):
    logger.info("Starting RandomForest hyperparameter tuning with Stratified CV...")

    study_rf = optuna.create_study(direction="maximize")
    study_rf.optimize(objective_rf, n_trials=10)

    # Log best CV result
    mlflow.log_params(study_rf.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_rf.best_value))

    # Train final RF model on full training set with best params
    best_rf = RandomForestClassifier(
        **study_rf.best_params,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    )
    best_rf.fit(X_train, y_train)

    # Final evaluation on untouched test set
    y_test_proba = best_rf.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_metrics = {
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "test_log_loss": float(log_loss(y_test, y_test_proba)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
    }

    mlflow.log_metrics(test_metrics)
    mlflow.sklearn.log_model(best_rf, name="best_rf_prauc_model")

    overall_results["RandomForest"] = {
        "model": best_rf,
        "cv_pr_auc": float(study_rf.best_value),
        "test_pr_auc": float(test_metrics["test_pr_auc"]),
        "params": study_rf.best_params,
    }

    print("-" * 40)
    print("RandomForest tuning completed!")
    print(f"Best CV PR-AUC: {study_rf.best_value:.4f}")
    print(f"Test PR-AUC: {test_metrics['test_pr_auc']:.4f}")
    print(f"Best Parameters: {study_rf.best_params}")

2026-04-11 15:41:14 [INFO] Starting RandomForest hyperparameter tuning with Stratified CV...
[I 2026-04-11 15:41:14,832] A new study created in memory with name: no-name-ffea8969-186f-4643-bcfe-1f45320de67b
[I 2026-04-11 15:41:24,690] Trial 0 finished with value: 0.23492892228870488 and parameters: {'n_estimators': 131, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 2, 'criterion': 'entropy'}. Best is trial 0 with value: 0.23492892228870488.


🏃 View run shivering-dove-897 at: http://127.0.0.1:5000/#/experiments/1/runs/2f2c5598c78b4653ac068db9f1154bc5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 15:41:40,298] Trial 1 finished with value: 0.235396727594813 and parameters: {'n_estimators': 249, 'max_depth': 12, 'min_samples_split': 4, 'min_samples_leaf': 3, 'criterion': 'entropy'}. Best is trial 1 with value: 0.235396727594813.


🏃 View run trusting-panda-602 at: http://127.0.0.1:5000/#/experiments/1/runs/18d7a3118d114df1851c2d64b246fb0c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 15:42:11,450] Trial 2 finished with value: 0.2350621529370029 and parameters: {'n_estimators': 490, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 3, 'criterion': 'gini'}. Best is trial 1 with value: 0.235396727594813.


...

🏃 View run debonair-auk-500 at: http://127.0.0.1:5000/#/experiments/1/runs/d710267aaa4e488fa8f433a54719a1ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 15:43:17,247] Trial 9 finished with value: 0.2361526209721313 and parameters: {'n_estimators': 292, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 5, 'criterion': 'entropy'}. Best is trial 7 with value: 0.23876274902426903.


🏃 View run skittish-elk-151 at: http://127.0.0.1:5000/#/experiments/1/runs/d31377c440b94e89a544698859d1f3f5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/04/11 15:43:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


----------------------------------------
RandomForest tuning completed!
Best CV PR-AUC: 0.2388
Test PR-AUC: 0.2405
Best Parameters: {'n_estimators': 207, 'max_depth': 9, 'min_samples_split': 7, 'min_samples_leaf': 5, 'criterion': 'entropy'}
🏃 View run RandomForest_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/1/runs/0dc04227e2124669b277de436fcc0284
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Lấy độ quan trọng của các feature từ model tốt nhất
# importances = best_model.feature_importances_
# feature_names = X_train.columns

# # 2. Tạo DataFrame để dễ sắp xếp
# feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# # 3. In ra Top 5
# print("Top 5 Important Features:")
# print(feature_importance_df.head(5))

# # 4. Vẽ biểu đồ trực quan
# plt.figure(figsize=(10, 6))
# plt.barh(feature_importance_df['Feature'].head(5), feature_importance_df['Importance'].head(5), color='skyblue')
# plt.xlabel('Importance Score')
# plt.title('Top 5 Features causing potential Data Leakage')
# plt.gca().invert_yaxis()
# plt.show()

### XGBOOST

In [12]:
# =========================================
# XGBoost tuning with CV
# =========================================

# Class imbalance ratio for positive class weighting
ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

def objective_xgb(trial):
    """
    Tune XGBoost using cross-validated PR-AUC on the training set only.
    This avoids test leakage during hyperparameter search.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 5, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio * 1.5),
        "random_state": RANDOM_STATE,
        "eval_metric": "logloss",
        "n_jobs": -1,
    }

    model = XGBClassifier(**params)

    # Out-of-fold predicted probabilities on training data
    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Default classification threshold for reporting auxiliary metrics
    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train, y_pred_oof)),
        "cv_precision": float(precision_score(y_train, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train, y_pred_oof, zero_division=0)),
    }

    # Log each Optuna trial as a nested MLflow run
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="XGBoost_PR_AUC_Optimization_CV"):
    logger.info("Starting XGBoost hyperparameter tuning with Stratified CV...")

    study_xgb = optuna.create_study(direction="maximize")
    study_xgb.optimize(objective_xgb, n_trials=50)

    # Log best CV result
    mlflow.log_params(study_xgb.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_xgb.best_value))

    # Train final XGBoost model on full training set with best params
    best_xgb = XGBClassifier(
        **study_xgb.best_params,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=-1,
    )
    best_xgb.fit(X_train, y_train)

    # Final evaluation on untouched test set
    y_test_proba = best_xgb.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_metrics = {
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "test_log_loss": float(log_loss(y_test, y_test_proba)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
    }

    mlflow.log_metrics(test_metrics)
    mlflow.xgboost.log_model(best_xgb, name="best_xgb_prauc_model")

    overall_results["XGBoost"] = {
        "model": best_xgb,
        "cv_pr_auc": float(study_xgb.best_value),
        "test_pr_auc": float(test_metrics["test_pr_auc"]),
        "params": study_xgb.best_params,
    }

    print("-" * 40)
    print("XGBoost tuning completed!")
    print(f"Best CV PR-AUC: {study_xgb.best_value:.4f}")
    print(f"Test PR-AUC: {test_metrics['test_pr_auc']:.4f}")
    print(f"Best Parameters: {study_xgb.best_params}")

2026-04-11 15:52:41 [INFO] Starting XGBoost hyperparameter tuning with Stratified CV...
[I 2026-04-11 15:52:41,842] A new study created in memory with name: no-name-933a6a95-9ff4-41ca-a59b-c46fc3ce603e
[I 2026-04-11 15:52:58,222] Trial 0 finished with value: 0.2193665550393386 and parameters: {'n_estimators': 728, 'max_depth': 6, 'learning_rate': 0.04903112834352123, 'subsample': 0.6782584273320265, 'colsample_bytree': 0.982378206708919, 'gamma': 4.3206981309872186, 'scale_pos_weight': 4.769873010491647}. Best is trial 0 with value: 0.2193665550393386.


🏃 View run clumsy-goose-896 at: http://127.0.0.1:5000/#/experiments/1/runs/591619cae34e44fba7f9da5b98ed669c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 15:53:26,119] Trial 1 finished with value: 0.2185396933424104 and parameters: {'n_estimators': 918, 'max_depth': 9, 'learning_rate': 0.03294265407620401, 'subsample': 0.561177852788644, 'colsample_bytree': 0.5498096547373519, 'gamma': 0.3292456536622107, 'scale_pos_weight': 6.003112670548729}. Best is trial 0 with value: 0.2193665550393386.


🏃 View run placid-fox-929 at: http://127.0.0.1:5000/#/experiments/1/runs/0f3f176f830c48ebae0e79323907bf8d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 15:53:36,415] Trial 2 finished with value: 0.22280481932327512 and parameters: {'n_estimators': 377, 'max_depth': 6, 'learning_rate': 0.060874814794266506, 'subsample': 0.6078343869143698, 'colsample_bytree': 0.6842828032990894, 'gamma': 3.6029381149497675, 'scale_pos_weight': 6.99580111313232}. Best is trial 2 with value: 0.22280481932327512.


...

🏃 View run bedecked-auk-915 at: http://127.0.0.1:5000/#/experiments/1/runs/ad3368b9ff3840a59ee05335d1c49b03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:00:50,368] Trial 48 finished with value: 0.23412977589874842 and parameters: {'n_estimators': 747, 'max_depth': 14, 'learning_rate': 0.19842015227428594, 'subsample': 0.8584298239881051, 'colsample_bytree': 0.7122781646317752, 'gamma': 4.76038792585919, 'scale_pos_weight': 1.0074767490574341}. Best is trial 40 with value: 0.23656676723876732.


🏃 View run valuable-mule-509 at: http://127.0.0.1:5000/#/experiments/1/runs/8c8871b6085e460c90debda13d86229a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:01:00,737] Trial 49 finished with value: 0.2175546446361968 and parameters: {'n_estimators': 836, 'max_depth': 11, 'learning_rate': 0.13390127816381694, 'subsample': 0.6798938289835931, 'colsample_bytree': 0.5621367140042131, 'gamma': 3.7099864983752515, 'scale_pos_weight': 3.3048393242981984}. Best is trial 40 with value: 0.23656676723876732.


🏃 View run casual-bug-151 at: http://127.0.0.1:5000/#/experiments/1/runs/027671faa3da4acdb429d675ceef4ec0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
----------------------------------------
XGBoost tuning completed!
Best CV PR-AUC: 0.2366
Test PR-AUC: 0.2351
Best Parameters: {'n_estimators': 864, 'max_depth': 7, 'learning_rate': 0.07404870294862903, 'subsample': 0.8920832774235486, 'colsample_bytree': 0.5456467971931089, 'gamma': 4.65970500666878, 'scale_pos_weight': 1.370916242370779}
🏃 View run XGBoost_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/1/runs/3eb1e97d58a5486694e662bf964301f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
# # Vẽ ma trận nhầm lẫn chuẩn hóa (normalized) theo hàng ngang (true labels)
# fig, ax = plt.subplots(figsize=(8, 6))
# disp_norm = ConfusionMatrixDisplay.from_estimator(
#     best_xgb,
#     X_test,
#     y_test,
#     display_labels=["Active (0)", "Churn (1)"],
#     cmap="Greens",
#     normalize='true', # Quan trọng: tính % theo từng lớp thực tế
#     ax=ax
# )

# plt.title("Normalized Confusion Matrix (%)")
# plt.show()

### LIGHTGBM

In [14]:
# =========================================
# LightGBM tuning with CV
# =========================================
def objective_lgbm(trial):
    """
    Tune LightGBM using cross-validated PR-AUC on the training set only.
    This avoids test leakage during hyperparameter search.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio * 1.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "verbosity": -1,
        "n_jobs": -1,
    }

    model = LGBMClassifier(**params)

    # Out-of-fold predicted probabilities on training data
    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    # Default classification threshold for reporting auxiliary metrics
    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train, y_pred_oof)),
        "cv_precision": float(precision_score(y_train, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train, y_pred_oof, zero_division=0)),
    }

    # Log each Optuna trial as a nested MLflow run
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="LGBM_PR_AUC_Optimization_CV"):
    logger.info("Starting LightGBM hyperparameter tuning with Stratified CV...")

    study_lgbm = optuna.create_study(direction="maximize")
    study_lgbm.optimize(objective_lgbm, n_trials=50)

    # Log best CV result
    mlflow.log_params(study_lgbm.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_lgbm.best_value))

    # Train final LightGBM model on full training set with best params
    best_lgbm = LGBMClassifier(
        **study_lgbm.best_params,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1,
    )
    best_lgbm.fit(X_train, y_train)

    # Final evaluation on untouched test set
    y_test_proba = best_lgbm.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_metrics = {
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "test_log_loss": float(log_loss(y_test, y_test_proba)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
    }

    mlflow.log_metrics(test_metrics)
    mlflow.lightgbm.log_model(best_lgbm, name="best_lgbm_prauc_model")

    overall_results["LightGBM"] = {
        "model": best_lgbm,
        "cv_pr_auc": float(study_lgbm.best_value),
        "test_pr_auc": float(test_metrics["test_pr_auc"]),
        "params": study_lgbm.best_params,
    }

    print("-" * 40)
    print("LightGBM tuning completed!")
    print(f"Best CV PR-AUC: {study_lgbm.best_value:.4f}")
    print(f"Test PR-AUC: {test_metrics['test_pr_auc']:.4f}")
    print(f"Best Parameters: {study_lgbm.best_params}")

2026-04-11 16:11:11 [INFO] Starting LightGBM hyperparameter tuning with Stratified CV...
[I 2026-04-11 16:11:11,461] A new study created in memory with name: no-name-df7fd5f7-5435-4b4b-ae6c-606721562cc1
[I 2026-04-11 16:11:57,182] Trial 0 finished with value: 0.21097507225828804 and parameters: {'n_estimators': 1067, 'learning_rate': 0.08264823360870166, 'num_leaves': 132, 'max_depth': 7, 'min_child_samples': 25, 'scale_pos_weight': 3.373673084863828, 'reg_alpha': 0.9122961235009924, 'reg_lambda': 0.011247157977812259}. Best is trial 0 with value: 0.21097507225828804.


🏃 View run unequaled-hawk-345 at: http://127.0.0.1:5000/#/experiments/1/runs/cadf13c58d664377bc233fa15df8b859
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:12:21,071] Trial 1 finished with value: 0.23041814969168448 and parameters: {'n_estimators': 1117, 'learning_rate': 0.031716954893400685, 'num_leaves': 32, 'max_depth': 4, 'min_child_samples': 79, 'scale_pos_weight': 6.748875722002325, 'reg_alpha': 2.2512233207784185, 'reg_lambda': 0.1839614146787839}. Best is trial 1 with value: 0.23041814969168448.


🏃 View run smiling-goose-464 at: http://127.0.0.1:5000/#/experiments/1/runs/056af7114688415a9c85b848f4a7e951
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:12:47,581] Trial 2 finished with value: 0.23639042295652035 and parameters: {'n_estimators': 423, 'learning_rate': 0.005585042900747331, 'num_leaves': 131, 'max_depth': 9, 'min_child_samples': 30, 'scale_pos_weight': 3.955036792216239, 'reg_alpha': 3.8126894004910596, 'reg_lambda': 0.001152336396292018}. Best is trial 2 with value: 0.23639042295652035.


...

🏃 View run dashing-bird-571 at: http://127.0.0.1:5000/#/experiments/1/runs/573fd3f19d744cb5b723f0d165e9300a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:27:44,322] Trial 49 finished with value: 0.2180573696872004 and parameters: {'n_estimators': 533, 'learning_rate': 0.09488323732793731, 'num_leaves': 124, 'max_depth': 6, 'min_child_samples': 62, 'scale_pos_weight': 5.735630917477187, 'reg_alpha': 6.662070082391561, 'reg_lambda': 1.7280680355859352}. Best is trial 47 with value: 0.23755670775745005.


🏃 View run gregarious-fox-63 at: http://127.0.0.1:5000/#/experiments/1/runs/d1971a022b9347ffbc8974f3c01ca67b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/04/11 16:27:46 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


----------------------------------------
LightGBM tuning completed!
Best CV PR-AUC: 0.2376
Test PR-AUC: 0.2326
Best Parameters: {'n_estimators': 415, 'learning_rate': 0.0057493843171183955, 'num_leaves': 118, 'max_depth': 5, 'min_child_samples': 63, 'scale_pos_weight': 5.4472360473324875, 'reg_alpha': 2.9175318036565674, 'reg_lambda': 2.7253620635345914}
🏃 View run LGBM_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/1/runs/9ef318856924403dbfed4624cbfeaa65
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Catboost

In [ ]:
# =========================================
# CatBoost tuning with CV
# =========================================
def objective_catboost(trial):
    """
    Tune CatBoost using cross-validated PR-AUC on the training set only.
    This avoids test leakage during hyperparameter search.
    """
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1000),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, ratio * 1.5),
        "random_state": RANDOM_STATE,
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "verbose": 0,
        "allow_writing_files": False,   #####
        "train_dir": str(Path("../artifacts/tmp/catboost_info"))
    }

    model = CatBoostClassifier(**params)

    # Out-of-fold predicted probabilities on training data
    y_proba_oof = cross_val_predict(
        estimator=model,
        X=X_train,
        y=y_train,
        cv=cv,
        method="predict_proba",
    )[:, 1]

    # Default classification threshold for reporting auxiliary metrics
    y_pred_oof = (y_proba_oof >= 0.5).astype(int)

    metrics = {
        "cv_pr_auc": float(average_precision_score(y_train, y_proba_oof)),
        "cv_roc_auc": float(roc_auc_score(y_train, y_proba_oof)),
        "cv_log_loss": float(log_loss(y_train, y_proba_oof)),
        "cv_accuracy": float(accuracy_score(y_train, y_pred_oof)),
        "cv_precision": float(precision_score(y_train, y_pred_oof, zero_division=0)),
        "cv_recall": float(recall_score(y_train, y_pred_oof, zero_division=0)),
        "cv_f1_score": float(f1_score(y_train, y_pred_oof, zero_division=0)),
    }

    # Log each Optuna trial as a nested MLflow run
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

    return metrics["cv_pr_auc"]


with mlflow.start_run(run_name="CatBoost_PR_AUC_Optimization_CV"):
    logger.info("Starting CatBoost hyperparameter tuning with Stratified CV...")

    study_catboost = optuna.create_study(direction="maximize")
    study_catboost.optimize(objective_catboost, n_trials=30)

    # Log best CV result
    mlflow.log_params(study_catboost.best_params)
    mlflow.log_metric("best_cv_pr_auc", float(study_catboost.best_value))

    # Train final CatBoost model on full training set with best params
    best_catboost = CatBoostClassifier(
        **study_catboost.best_params,
        random_state=RANDOM_STATE,
        loss_function="Logloss",
        eval_metric="PRAUC",
        verbose=0,
        allow_writing_files=False,
    )
    best_catboost.fit(X_train, y_train)

    # Final evaluation on untouched test set
    y_test_proba = best_catboost.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= 0.5).astype(int)

    test_metrics = {
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "test_roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "test_log_loss": float(log_loss(y_test, y_test_proba)),
        "test_accuracy": float(accuracy_score(y_test, y_test_pred)),
        "test_precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "test_recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "test_f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
    }

    mlflow.log_metrics(test_metrics)
    mlflow.sklearn.log_model(best_catboost, name="best_catboost_prauc_model")

    overall_results["CatBoost"] = {
        "model": best_catboost,
        "cv_pr_auc": float(study_catboost.best_value),
        "test_pr_auc": float(test_metrics["test_pr_auc"]),
        "params": study_catboost.best_params,
    }

    print("-" * 40)
    print("CatBoost tuning completed!")
    print(f"Best CV PR-AUC: {study_catboost.best_value:.4f}")
    print(f"Test PR-AUC: {test_metrics['test_pr_auc']:.4f}")
    print(f"Best Parameters: {study_catboost.best_params}")

2026-04-11 16:40:20 [INFO] Starting CatBoost hyperparameter tuning with Stratified CV...
[I 2026-04-11 16:40:20,950] A new study created in memory with name: no-name-69d6d53d-610b-4b4f-a695-2937d12bbfd1
[I 2026-04-11 16:41:24,784] Trial 0 finished with value: 0.21973265901210937 and parameters: {'iterations': 698, 'depth': 7, 'learning_rate': 0.09141125011342928, 'l2_leaf_reg': 8.278237754362989, 'subsample': 0.9275477415734181, 'random_strength': 1.4579462851515002, 'scale_pos_weight': 2.036926994296398}. Best is trial 0 with value: 0.21973265901210937.


🏃 View run funny-squid-626 at: http://127.0.0.1:5000/#/experiments/1/runs/a7bee7b5b49049688e89cecd2782d4a1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:42:10,878] Trial 1 finished with value: 0.230854765500619 and parameters: {'iterations': 631, 'depth': 5, 'learning_rate': 0.06697946186303694, 'l2_leaf_reg': 3.8012222917645992, 'subsample': 0.9507105187964047, 'random_strength': 0.28871532598264305, 'scale_pos_weight': 2.353755671398708}. Best is trial 1 with value: 0.230854765500619.


🏃 View run worried-snail-34 at: http://127.0.0.1:5000/#/experiments/1/runs/55dc0a04ae4a49f1831e1f8eb76130a5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 16:42:46,155] Trial 2 finished with value: 0.23169144792346424 and parameters: {'iterations': 541, 'depth': 4, 'learning_rate': 0.10025323284144673, 'l2_leaf_reg': 7.335234952019772, 'subsample': 0.929769607466133, 'random_strength': 0.19340205257376009, 'scale_pos_weight': 4.955388698044528}. Best is trial 2 with value: 0.23169144792346424.


...

🏃 View run languid-boar-578 at: http://127.0.0.1:5000/#/experiments/1/runs/693ba061c6aa4462baeb2969ebb26925
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-04-11 17:20:02,675] Trial 29 finished with value: 0.23615612518454346 and parameters: {'iterations': 385, 'depth': 9, 'learning_rate': 0.010556700269666576, 'l2_leaf_reg': 7.102612825356804, 'subsample': 0.7181612761087698, 'random_strength': 2.7361589851608703, 'scale_pos_weight': 5.334539661254458}. Best is trial 26 with value: 0.23692242249695633.


🏃 View run flawless-grub-98 at: http://127.0.0.1:5000/#/experiments/1/runs/9b8d7107e6c5437a968473fc00f5423c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/04/11 17:20:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


----------------------------------------
CatBoost tuning completed!
Best CV PR-AUC: 0.2369
Test PR-AUC: 0.2444
Best Parameters: {'iterations': 436, 'depth': 9, 'learning_rate': 0.012553386061217049, 'l2_leaf_reg': 6.513097185041718, 'subsample': 0.6750746733431446, 'random_strength': 2.760553451583119, 'scale_pos_weight': 7.204809399002687}
🏃 View run CatBoost_PR_AUC_Optimization_CV at: http://127.0.0.1:5000/#/experiments/1/runs/0a26ae2c5eeb4296a23eafe799aefdee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [51]:
#### Backfill additional evaluation metrics for trained models

DEFAULT_THRESHOLD = 0.5

for model_name, info in overall_results.items():
    model = info["model"]
    y_test_proba = model.predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_proba >= DEFAULT_THRESHOLD).astype(int)

    info.update({
        "test_pr_auc": float(average_precision_score(y_test, y_test_proba)),
        "accuracy": float(accuracy_score(y_test, y_test_pred)),
        "precision": float(precision_score(y_test, y_test_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_test_pred, zero_division=0)),
        "f1_score": float(f1_score(y_test, y_test_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, y_test_proba)),
        "log_loss": float(log_loss(y_test, y_test_proba)),
        "predicted_positive_rate": float(y_test_pred.mean()),
        "threshold": float(DEFAULT_THRESHOLD),
    })

print("Updated overall_results with additional test metrics.")
for model_name, info in overall_results.items():
    print(
        model_name,
        "PR-AUC:", round(info["test_pr_auc"], 4),
        "| Acc:", round(info["accuracy"], 4),
        "| Prec:", round(info["precision"], 4),
        "| Rec:", round(info["recall"], 4),
        "| F1:", round(info["f1_score"], 4),
        "| ROC-AUC:", round(info["roc_auc"], 4),
        "| LogLoss:", round(info["log_loss"], 4),
        "| PPR:", round(info["predicted_positive_rate"], 4),
    )

Updated overall_results with additional test metrics.
RandomForest PR-AUC: 0.2405 | Acc: 0.6305 | Prec: 0.2544 | Rec: 0.5935 | F1: 0.3562 | ROC-AUC: 0.6395 | LogLoss: 0.6331 | PPR: 0.4017
XGBoost PR-AUC: 0.2351 | Acc: 0.8278 | Prec: 0.0 | Rec: 0.0 | F1: 0.0 | ROC-AUC: 0.6323 | LogLoss: 0.4495 | PPR: 0.0
LightGBM PR-AUC: 0.2326 | Acc: 0.6188 | Prec: 0.251 | Rec: 0.6115 | F1: 0.3559 | ROC-AUC: 0.6349 | LogLoss: 0.6531 | PPR: 0.4196
CatBoost PR-AUC: 0.2444 | Acc: 0.4142 | Prec: 0.2076 | Rec: 0.8525 | F1: 0.3339 | ROC-AUC: 0.6383 | LogLoss: 0.7648 | PPR: 0.7072
Baseline_LogisticRegression PR-AUC: 0.2373 | Acc: 0.6208 | Prec: 0.2514 | Rec: 0.608 | F1: 0.3558 | ROC-AUC: 0.6344 | LogLoss: 0.6614 | PPR: 0.4164


# Final summary and best model export

In [53]:
# =========================================
# Final summary and best model export
# =========================================

# Convert overall results to a summary table
summary_rows = []
for model_name, info in overall_results.items():
    summary_rows.append({
        "model": model_name,
        "cv_pr_auc": None if info["cv_pr_auc"] is None else float(info["cv_pr_auc"]),
        "test_pr_auc": float(info["test_pr_auc"]),
        "accuracy": float(info["accuracy"]),
        "precision": float(info["precision"]),
        "recall": float(info["recall"]),
        "f1_score": float(info["f1_score"]),
        "roc_auc": float(info["roc_auc"]),
        "log_loss": float(info["log_loss"]),
        "predicted_positive_rate": float(info["predicted_positive_rate"]),
        "threshold": float(info["threshold"]),
        "params": info["params"],
    })

results_df = pd.DataFrame(summary_rows).sort_values(
    by="test_pr_auc",
    ascending=False
).reset_index(drop=True)

print("=" * 100)
print("MODEL COMPARISON SUMMARY")
print("=" * 100)
display(
    results_df[
        [
            "model",
            "cv_pr_auc",
            "test_pr_auc",
            "accuracy",
            "precision",
            "recall",
            "f1_score",
            "roc_auc",
            "log_loss",
            "predicted_positive_rate",
            "threshold",
        ]
    ]
)

# Select the current best model based on clean test PR-AUC
best_model_name = results_df.loc[0, "model"]
best_info = overall_results[best_model_name]

print("\n" + "=" * 60)
print("CURRENT WINNER")
print("=" * 60)
print(f"Best model: {best_model_name}")
print(f"Best CV PR-AUC: {best_info['cv_pr_auc']:.4f}")
print(f"Best Test PR-AUC: {best_info['test_pr_auc']:.4f}")
print(f"Best Accuracy: {best_info['accuracy']:.4f}")
print(f"Best Precision: {best_info['precision']:.4f}")
print(f"Best Recall: {best_info['recall']:.4f}")
print(f"Best F1-score: {best_info['f1_score']:.4f}")
print(f"Best ROC-AUC: {best_info['roc_auc']:.4f}")
print(f"Best Log Loss: {best_info['log_loss']:.4f}")
print(f"Predicted Positive Rate: {best_info['predicted_positive_rate']:.4f}")
print(f"Threshold: {best_info['threshold']:.2f}")
print(f"Best Parameters: {best_info['params']}")


MODEL COMPARISON SUMMARY


,model,cv_pr_auc,test_pr_auc,accuracy,precision,recall,f1_score,roc_auc,log_loss,predicted_positive_rate,threshold
0,CatBoost,0.236922,0.244381,0.4142,0.207579,0.852497,0.333864,0.638279,0.764783,0.7072,0.5
1,RandomForest,0.238763,0.240469,0.6305,0.254419,0.593496,0.356160,0.639520,0.633092,0.4017,0.5
2,Baseline_LogisticRegression,NaN,0.237302,0.6208,0.251441,0.608014,0.355759,0.634450,0.661404,0.4164,0.5
3,XGBoost,0.236567,0.235103,0.8278,0.000000,0.000000,0.000000,0.632304,0.449477,0.0000,0.5
4,LightGBM,0.237557,0.232556,0.6188,0.250953,0.611498,0.355863,0.634938,0.653060,0.4196,0.5



CURRENT WINNER
Best model: CatBoost
Best CV PR-AUC: 0.2369
Best Test PR-AUC: 0.2444
Best Accuracy: 0.4142
Best Precision: 0.2076
Best Recall: 0.8525
Best F1-score: 0.3339
Best ROC-AUC: 0.6383
Best Log Loss: 0.7648
Predicted Positive Rate: 0.7072
Threshold: 0.50
Best Parameters: {'iterations': 436, 'depth': 9, 'learning_rate': 0.012553386061217049, 'l2_leaf_reg': 6.513097185041718, 'subsample': 0.6750746733431446, 'random_strength': 2.760553451583119, 'scale_pos_weight': 7.204809399002687}


In [54]:
### Threshold Selection for Final Model
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the selected winner model directly from overall_results
final_model = overall_results[best_model_name]["model"]

# Predict probabilities on the same clean test set
y_proba_final = final_model.predict_proba(X_test)[:, 1]

# Sweep thresholds
rows = []
thresholds = np.arange(0.10, 0.91, 0.05)

for t in thresholds:
    y_pred_final = (y_proba_final >= t).astype(int)
    rows.append({
        "threshold": round(float(t), 2),
        "accuracy": float(accuracy_score(y_test, y_pred_final)),
        "precision": float(precision_score(y_test, y_pred_final, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred_final, zero_division=0)),
        "f1_score": float(f1_score(y_test, y_pred_final, zero_division=0)),
        "predicted_positive_rate": float(y_pred_final.mean()),
    })

threshold_df = pd.DataFrame(rows).sort_values(
    by="f1_score",
    ascending=False
).reset_index(drop=True)

print("=" * 100)
print(f"THRESHOLD ANALYSIS FOR FINAL MODEL: {best_model_name}")
print("=" * 100)
display(threshold_df)

THRESHOLD ANALYSIS FOR FINAL MODEL: CatBoost


,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate
0,0.60,0.6289,0.254262,0.597561,0.356734,0.4047
1,0.65,0.6408,0.256129,0.570267,0.353492,0.3834
2,0.55,0.5424,0.226942,0.688734,0.341393,0.5226
3,0.50,0.4142,0.207579,0.852497,0.333864,0.7072
4,0.45,0.3861,0.204628,0.888502,0.332645,0.7477
5,0.40,0.3738,0.201943,0.893148,0.329407,0.7616
6,0.35,0.3027,0.190426,0.937863,0.316574,0.8481
7,0.30,0.2009,0.175080,0.980836,0.297124,0.9647
8,0.20,0.1723,0.172217,1.000000,0.293832,0.9999
9,0.10,0.1722,0.172200,1.000000,0.293807,1.0000


In [55]:
## Recall-priority threshold candidates

recall_priority_df = threshold_df[threshold_df["recall"] >= 0.80].sort_values(
    by=["precision", "f1_score"],
    ascending=[False, False]
).reset_index(drop=True)

print("=" * 100)
print("THRESHOLDS WITH RECALL >= 0.80")
print("=" * 100)
display(recall_priority_df.head(10))

THRESHOLDS WITH RECALL >= 0.80


,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate
0,0.50,0.4142,0.207579,0.852497,0.333864,0.7072
1,0.45,0.3861,0.204628,0.888502,0.332645,0.7477
2,0.40,0.3738,0.201943,0.893148,0.329407,0.7616
3,0.35,0.3027,0.190426,0.937863,0.316574,0.8481
4,0.30,0.2009,0.175080,0.980836,0.297124,0.9647
5,0.20,0.1723,0.172217,1.000000,0.293832,0.9999
6,0.10,0.1722,0.172200,1.000000,0.293807,1.0000
7,0.15,0.1722,0.172200,1.000000,0.293807,1.0000
8,0.25,0.1738,0.172082,0.996516,0.293484,0.9972


In [56]:
### Select final threshold
# Final threshold decision based on business preference
chosen_threshold = 0.50

chosen_threshold_row = threshold_df[threshold_df["threshold"] == round(chosen_threshold, 2)]

print("=" * 100)
print("FINAL THRESHOLD DECISION")
print("=" * 100)
print(f"Chosen threshold: {chosen_threshold}")

if not chosen_threshold_row.empty:
    display(chosen_threshold_row)
else:
    print("Chosen threshold not found in threshold table.")

FINAL THRESHOLD DECISION
Chosen threshold: 0.5


,threshold,accuracy,precision,recall,f1_score,predicted_positive_rate
3,0.5,0.4142,0.207579,0.852497,0.333864,0.7072


# Export final configs and reports

In [ ]:
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts/models")
REPORTS_DIR = Path("../reports")

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# Save model comparison report
comparison_df = results_df.copy().rename(columns={"model": "model_name"})
comparison_path = REPORTS_DIR / "model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

print("\nSaved model comparison report to:")
print(comparison_path)

# Build final best model config WITH chosen threshold
best_model_config = {
    "project": "Netflix_Prediction",
    "best_model_overall": best_model_name,
    "selection_metric": "test_pr_auc",
    "threshold": float(chosen_threshold),
    "metrics": {
        "cv_pr_auc": float(best_info["cv_pr_auc"]),
        "test_pr_auc": float(best_info["test_pr_auc"]),
        "accuracy": float(best_info["accuracy"]),
        "precision": float(best_info["precision"]),
        "recall": float(best_info["recall"]),
        "f1_score": float(best_info["f1_score"]),
        "roc_auc": float(best_info["roc_auc"]),
        "log_loss": float(best_info["log_loss"]),
        "predicted_positive_rate": float(best_info["predicted_positive_rate"]),
    },
    "hyperparameters": best_info["params"],
}

best_model_config_path = ARTIFACTS_DIR / "best_model_config.yaml"
with open(best_model_config_path, "w", encoding="utf-8") as f:
    yaml.dump(best_model_config, f, default_flow_style=False, sort_keys=False)

print("\nSaved best model config to:")
print(best_model_config_path)


###
baseline_model_name = "Baseline_LogisticRegression"
baseline_info = overall_results[baseline_model_name]

baseline_config = {
    "project": "Netflix_Prediction",
    "model_name": baseline_model_name,
    "selection_metric": "test_pr_auc",
    "threshold": float(baseline_info.get("threshold", 0.5)),
    "metrics": {
        "cv_pr_auc": None if baseline_info["cv_pr_auc"] is None else float(baseline_info["cv_pr_auc"]),
        "test_pr_auc": float(baseline_info["test_pr_auc"]),
        "accuracy": float(baseline_info["accuracy"]),
        "precision": float(baseline_info["precision"]),
        "recall": float(baseline_info["recall"]),
        "f1_score": float(baseline_info["f1_score"]),
        "roc_auc": float(baseline_info["roc_auc"]),
        "log_loss": float(baseline_info["log_loss"]),
        "predicted_positive_rate": float(baseline_info["predicted_positive_rate"]),
    },
    "hyperparameters": baseline_info["params"],
}

baseline_config_path = ARTIFACTS_DIR / "baseline_model_config.yaml"
with open(baseline_config_path, "w", encoding="utf-8") as f:
    yaml.dump(baseline_config, f, default_flow_style=False, sort_keys=False)

print("\nSaved baseline model config to:")
print(baseline_config_path)


Saved model comparison report to:
..\reports\model_comparison.csv

Saved best model config to:
..\artifacts\models\best_model_config.yaml

Saved baseline model config to:
..\artifacts\models\baseline_model_config.yaml
